# Семинар 2. Своя среда в Gymnasium: GridWorld, обёртки, Cross-Entropy

План:

1. Каркас среды: пространства, `reset`, `step` — заполняем по шагам
2. Проверка `check_env`, текстовый и графический `render`
3. Случайный и ручной агенты
4. Обёртки: лимит шагов, своя награда, статистика эпизодов
5. Cross-Entropy из недели 1 на своей среде; что происходит на скользком полу
6. Что дальше: домашнее задание

Ячейки с `# TODO` заполняете сами; ниже каждой — проверка, которая должна пройти.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

## 1. Каркас среды

Договоримся о карте: строки, `S` — старт, `G` — цель, `#` — стена, `.` — пол. Состояние — номер клетки `row * n_cols + col`, действия `0..3` = ←, ↓, →, ↑. С вероятностью `slip` действие заменяется на случайное. Награда: 1 при приходе в цель, иначе 0; эпизод заканчивается в цели.

Заполните `reset` и `step`. Подсказки:

* `super().reset(seed=seed)` создаёт `self.np_random` — используйте **его**, а не `np.random`, иначе среда не будет воспроизводимой;
* шаг в стену или за границу оставляет агента на месте;
* `step` возвращает ровно пять значений: `obs, reward, terminated, truncated, info`.

In [ ]:
class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["ansi", "rgb_array"], "render_fps": 4}
    MOVES = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}   # ←, ↓, →, ↑  как (dr, dc)
    ARROWS = "←↓→↑"
    DEFAULT_LAYOUT = ["S....#", ".##..#", "...#..", ".#..#.", ".#.#..", "....#G"]

    def __init__(self, layout=None, slip=0.0, render_mode=None):
        self.layout = [list(row) for row in (layout or self.DEFAULT_LAYOUT)]
        self.n_rows, self.n_cols = len(self.layout), len(self.layout[0])
        self.slip = slip
        self.render_mode = render_mode
        self.observation_space = spaces.Discrete(self.n_rows * self.n_cols)
        self.action_space = spaces.Discrete(4)
        self.start = self._find("S")
        self.goal = self._find("G")
        self.pos = self.start

    def _find(self, char):
        for r, row in enumerate(self.layout):
            for c, cell in enumerate(row):
                if cell == char:
                    return (r, c)
        raise ValueError(f"на карте нет клетки {char!r}")

    def _obs(self):
        return self.pos[0] * self.n_cols + self.pos[1]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO: вернуть агента на старт и вернуть (obs, info)
        raise NotImplementedError

    def step(self, action):
        # TODO: 1) с вероятностью self.slip заменить action случайным (через self.np_random)
        #       2) сдвинуть агента, если клетка свободна и внутри поля
        #       3) terminated = агент в цели; reward = 1.0 если terminated иначе 0.0
        #       4) вернуть (obs, reward, terminated, False, {})
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return "\n".join("".join("A" if (r, c) == self.pos else ch for c, ch in enumerate(row))
                             for r, row in enumerate(self.layout))
        if self.render_mode == "rgb_array":
            return self._render_rgb()

    def _render_rgb(self):
        colors = {".": "#f4f4f4", "#": "#555555", "S": "#cfe3f7", "G": "#a9dfa9"}
        fig, ax = plt.subplots(figsize=(2.6, 2.6 * self.n_rows / self.n_cols))
        for r, row in enumerate(self.layout):
            for c, ch in enumerate(row):
                ax.add_patch(plt.Rectangle((c, r), 1, 1, color=colors[ch], ec="white"))
        ax.add_patch(plt.Circle((self.pos[1] + 0.5, self.pos[0] + 0.5), 0.3, color="#4C72B0"))
        ax.set_xlim(0, self.n_cols); ax.set_ylim(self.n_rows, 0); ax.set_aspect("equal"); ax.axis("off")
        fig.tight_layout(pad=0); fig.canvas.draw()
        img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
        plt.close(fig)
        return img

In [ ]:
# Проверка 1: базовое поведение.
env = GridWorldEnv()
obs, info = env.reset(seed=0)
assert obs == 0 and isinstance(info, dict)
obs, r, term, trunc, info = env.step(2)          # → из старта
assert obs == 1 and r == 0.0 and not term and not trunc
obs, *_ = env.step(3)                            # ↑ в стену поля: остаёмся
assert obs == 1
env.reset(seed=0)
obs, *_ = env.step(1); obs, *_ = env.step(1)     # ↓ ↓
assert obs == 12, obs
print("OK: reset/step ведут себя правильно")

## 2. Проверка интерфейса и отрисовка

`check_env` из Gymnasium — обязательный первый тест любой среды. Он ловит неправильные типы, невоспроизводимый `seed`, лишние значения из `step`.

In [ ]:
check_env(GridWorldEnv())
print("check_env: ok")

env = GridWorldEnv(render_mode="ansi")
env.reset(seed=0)
print(env.render())

env = GridWorldEnv(render_mode="rgb_array")
env.reset(seed=0)
plt.imshow(env.render()); plt.axis("off"); plt.show()

## 3. Случайный и ручной агенты

Случайный агент — базовая линия: любая политика, которую мы обучим, должна быть лучше него. Ручной агент — таблица «клетка → действие», написанная руками: убедитесь, что вы сами умеете решать задачу, прежде чем заставлять агента.

Заполните `manual_policy` так, чтобы агент доходил до цели на карте по умолчанию. Достаточно задать действия для клеток, через которые проходит маршрут.

In [ ]:
def run_episode(env, policy, seed=0, max_steps=100):
    """policy(obs) -> action. Возвращает (return, число шагов)."""
    obs, _ = env.reset(seed=seed)
    total = 0.0
    for t in range(max_steps):
        obs, r, terminated, truncated, _ = env.step(policy(obs))
        total += r
        if terminated or truncated:
            return total, t + 1
    return total, max_steps

rng = np.random.default_rng(0)
random_policy = lambda obs: int(rng.integers(4))

results = [run_episode(GridWorldEnv(), random_policy, seed=s) for s in range(200)]
print(f"случайный агент: доходит в {np.mean([g for g, _ in results]):.0%} эпизодов "
      f"(лимит 100 шагов), в среднем за {np.mean([t for _, t in results]):.0f} шагов")

# TODO: заполните маршрут (номер клетки -> действие 0..3). Карта:
#   S....#
#   .##..#
#   ...#..
#   .#..#.
#   .#.#..
#   ....#G
route = {
    0: 1,   # клетка 0 (старт): вниз
    # ...
}
manual_policy = lambda obs: route.get(int(obs), 0)

In [ ]:
# Проверка 2: ручной маршрут доходит до цели не более чем за 12 шагов.
G, steps = run_episode(GridWorldEnv(), manual_policy)
assert G == 1.0, "ручная политика не доходит до цели"
assert steps <= 12, f"маршрут слишком длинный: {steps} шагов (кратчайший — 10)"
print(f"OK: ручная политика доходит за {steps} шагов")

## 4. Обёртки

Обёртка держит внутри другую среду и меняет что-то по дороге. Готовые: `TimeLimit` (обрыв по шагам), `RecordEpisodeStatistics` (return и длина в `info["episode"]`). Свою пишут, наследуя `gym.Wrapper` или один из `gym.RewardWrapper` / `gym.ObservationWrapper` / `gym.ActionWrapper`.

**Задание:** напишите `StepPenalty` — обёртку, которая вычитает `penalty` из награды на каждом шаге (плотная награда «торопись»). Наследуйте `gym.RewardWrapper` и переопределите один метод `reward(self, r)`.

In [ ]:
from gymnasium.wrappers import TimeLimit, RecordEpisodeStatistics

class StepPenalty(gym.RewardWrapper):
    def __init__(self, env, penalty=0.01):
        super().__init__(env)
        self.penalty = penalty

    def reward(self, r):
        # TODO: вернуть награду с вычтенным штрафом
        raise NotImplementedError


env = RecordEpisodeStatistics(TimeLimit(StepPenalty(GridWorldEnv(), penalty=0.01), max_episode_steps=20))
obs, _ = env.reset(seed=0)
while True:
    obs, r, terminated, truncated, info = env.step(random_policy(obs))
    if terminated or truncated:
        break
print("terminated:", terminated, " truncated:", truncated)
print("info['episode']:", {k: float(v) for k, v in info["episode"].items()})

# Проверка 3
assert truncated or terminated
assert abs(float(info["episode"]["r"]) - (1.0 * terminated - 0.01 * float(info["episode"]["l"]))) < 1e-6
print("OK: обёртки работают")

## 5. Cross-Entropy на своей среде

Алгоритм из лекции 1 без изменений: ему нужны только `reset`, `step` и размеры пространств. Обучим на карте по умолчанию, потом посмотрим, что происходит на скользком полу (`slip=0.3`).

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей: состояния, действия, суммарная награда."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total


def cross_entropy_method(env, n_iter=25, n_sessions=200, q=0.7, laplace=0.5, mix=0.5,
                         seed=0, max_steps=100, evaluate=None):
    """Табличный Cross-Entropy из лекции 1. evaluate(policy) -> число, если хотим отдельную метрику."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    log = {"mean": [], "eval": []}
    for it in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        returns = np.array([G for _, _, G in sessions])
        threshold = np.quantile(returns, q)
        elite = [s for s in sessions if s[2] >= threshold and s[2] > returns.min()]
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, _ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
        log["mean"].append(returns.mean())
        if evaluate is not None:
            log["eval"].append(evaluate(policy))
    return policy, log


def show_policy(env, policy):
    for r in range(env.n_rows):
        print("  " + " ".join(env.layout[r][c] if env.layout[r][c] in "#G"
                              else env.ARROWS[int(np.argmax(policy[r * env.n_cols + c]))]
                              for c in range(env.n_cols)))

env = GridWorldEnv()
policy, log = cross_entropy_method(env, n_iter=20, n_sessions=200)
plt.plot(log["mean"], marker="."); plt.xlabel("итерация"); plt.ylabel("доля успехов"); plt.show()
show_policy(env, policy)

# Проверка 4: выученная (стохастическая) политика доходит до цели почти всегда.
rng = np.random.default_rng(5)
success = np.mean([run_session(GridWorldEnv(), policy, rng)[2] for _ in range(200)])
assert success >= 0.9, f"доля успехов {success:.2f} < 0.9"
print(f"OK: доля успехов выученной политики {success:.0%}")

In [ ]:
# Скользкий пол: действие с вероятностью 0.3 заменяется случайным.
fig, ax = plt.subplots(figsize=(7, 4))
for slip, color in [(0.0, "C0"), (0.3, "C3")]:
    for seed in range(3):
        _, log = cross_entropy_method(GridWorldEnv(slip=slip), n_iter=25, n_sessions=200, seed=seed)
        ax.plot(log["mean"], color=color, alpha=0.8, label=f"slip = {slip}" if seed == 0 else None)
ax.set_xlabel("итерация"); ax.set_ylabel("доля успехов"); ax.legend(); plt.show()

**Обсудите:** почему на скользком полу кривая ниже и шумнее, хотя цель всё та же? Что произойдёт, если сглаживание (`laplace`, `mix`) убрать? Какую награду вы бы добавили обёрткой, чтобы агент на скользком полу держался подальше от стен, — и как проверить, что она не сломала настоящую цель?

## 6. Что дальше

* **Домашнее задание** (`../homework/homework.ipynb`): своя среда «управление запасами» как `gym.Env` со случайным спросом, две версии награды и сравнение обучения Cross-Entropy, формальное описание среды как MDP.
* **Неделя 3**: методы, использующие структуру MDP, — ценность состояния, уравнение Беллмана, динамическое программирование, Monte-Carlo и TD-обучение.